In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Load database config from YAML
with open("../configs/database.yaml", "r") as f:
    config = yaml.safe_load(f)

pg_conf = config["database"]["postgres"]
pg_user = quote_plus(pg_conf["user"])
pg_password = quote_plus(pg_conf["password"])
pg_host = pg_conf["host"]
pg_port = pg_conf["port"]
pg_db = pg_conf["db"]

conn_str = (
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}"
)
engine = create_engine(conn_str)
print("Connected to PostgreSQL:", pg_conf['host'], pg_conf['db'])

Connected to PostgreSQL: localhost storage_analytics


In [ ]:
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

In [3]:
curated_df["workload_pattern"].value_counts()

workload_pattern
balanced             5464
saturated            1433
burst_io              981
read_heavy            771
high_throughput       625
latency_sensitive     385
write_heavy           252
small_io_pressure     169
Name: count, dtype: int64

In [4]:
anomaly_df.groupby("workload_pattern").size().sort_values(ascending=False)

workload_pattern
saturated            1338
balanced              605
latency_sensitive     390
high_throughput       144
write_heavy            95
read_heavy             83
burst_io               35
small_io_pressure      15
dtype: int64

In [5]:
anomaly_df["root_cause_hint"].value_counts()

root_cause_hint
Joint increase in latency and queue depth suggests pressure buildup    633
Composite saturation signal indicates elevated device stress           403
Latency anomaly detected without clear saturation signal               354
Queue depth anomaly detected                                           345
Multivariate anomaly aligned with saturation pattern                   278
Multivariate anomaly indicates unusual joint metric behavior           210
Queue buildup likely contributing to latency pressure                  155
High IOPS with small requests suggests random IO pressure               62
Latency degradation likely under write-heavy pressure                   61
IOPS anomaly indicates bursty load increase                             52
Latency spike likely driven by saturation and queue buildup             45
Device operating near full utilization                                  40
Latency spike observed during read-heavy demand                         34
Utilizati

In [6]:
anomaly_df[anomaly_df["severity"] == "critical"][
    ["device", "timestamp", "metric_name", "severity", "workload_pattern", "root_cause_hint"]
].sort_values("timestamp")

,device,timestamp,metric_name,severity,workload_pattern,root_cause_hint
1241,sdb,2026-03-28 18:31:36.173899+00:00,avg_latency_ms,critical,balanced,Latency anomaly detected without clear saturat...
1246,sdb,2026-03-28 21:46:38.801807+00:00,avg_latency_ms,critical,balanced,Latency anomaly detected without clear saturat...
0,dm-0,2026-03-28 21:55:49.203082+00:00,avg_latency_ms,critical,saturated,Latency anomaly detected without clear saturat...
318,dm-0,2026-03-28 21:55:49.203082+00:00,latency_pressure,critical,saturated,Joint increase in latency and queue depth sugg...
1716,sdb,2026-03-28 22:00:42.880861+00:00,saturation_score,critical,write_heavy,Composite saturation signal indicates elevated...
...,...,...,...,...,...,...
2199,sdb,2026-04-04 17:00:52.366825+00:00,latency_pressure,critical,balanced,Joint increase in latency and queue depth sugg...
1507,sdb,2026-04-04 17:00:52.366825+00:00,avg_latency_ms,critical,balanced,Latency anomaly detected without clear saturat...
1508,sdb,2026-04-04 17:05:51.271393+00:00,avg_latency_ms,critical,latency_sensitive,Latency degradation likely under write-heavy p...
1930,sdb,2026-04-04 17:05:51.271393+00:00,saturation_score,critical,latency_sensitive,Composite saturation signal indicates elevated...


In [7]:
anomaly_df.groupby(["device", "root_cause_hint"]).size().sort_values(ascending=False).head(20)

device   root_cause_hint                                                    
sdb      Joint increase in latency and queue depth suggests pressure buildup    270
         Latency anomaly detected without clear saturation signal               222
         Composite saturation signal indicates elevated device stress           219
         Multivariate anomaly indicates unusual joint metric behavior           210
         Multivariate anomaly aligned with saturation pattern                   149
dm-0     Queue depth anomaly detected                                           128
nvme1n1  Joint increase in latency and queue depth suggests pressure buildup    114
dm-0     Joint increase in latency and queue depth suggests pressure buildup    100
sdb      Queue buildup likely contributing to latency pressure                   99
nvme1n1  Queue depth anomaly detected                                            91
dm-0     Multivariate anomaly aligned with saturation pattern                    84